# Imports

In [1]:
import numpy as np
import pandas as pd
from _spo_utils import camel_to_snake, import_json
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

In [2]:
# INFO: Makes Pandas show all available columns in DataFrame
pd.set_option('display.max_columns', None)

# Constants

In [3]:
# PATH_SPOTIFY = '../../data/2_processed/Teixeira/final_df.csv'
PATH_SPOTIFY = '../../data/2_processed/consolidated.csv'
# PATH_PROCESSED = '../../data/2_processed/Teixeira/processed.csv'
PATH_PROCESSED = '../../data/2_processed/curated.csv'
PATH_LIBRARY = '../../data/1_raw/YourLibrary.json'
PATH_PLAYLIST = '../../data/1_raw/Playlist1.json'

# Loads consolidated DataFrame

In [4]:
final_df = pd.read_csv(PATH_SPOTIFY)
final_df.columns

Index(['Unnamed: 0', 'ts', 'platform', 'ms_played', 'conn_country', 'ip_addr',
       'master_metadata_track_name', 'master_metadata_album_album_name',
       'spotify_track_uri', 'episode_name', 'episode_show_name',
       'spotify_episode_uri', 'audiobook_title', 'audiobook_uri',
       'audiobook_chapter_uri', 'audiobook_chapter_title', 'reason_start',
       'reason_end', 'shuffle', 'skipped', 'offline', 'offline_timestamp',
       'incognito_mode', 'artist_name', 'segment', 'track_id', 'acousticness',
       'danceability', 'energy', 'instrumentalness', 'key', 'liveness',
       'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'popularity',
       'duration_ms', 'lat_long', 'temperature', 'precipitation'],
      dtype='object')

<h3 style='color:gray;'>Imports Library JSON file, using record_path (1/2)</h3>

In [5]:
library = import_json(PATH_LIBRARY, record_path=['tracks'])
library.head(1)

,artist,album,track,uri
0,Set It Off,Wolf In Sheep's Clothing [REBORN],Wolf In Sheep's Clothing [REBORN],spotify:track:1tpidJ4FBn9TwshePh1bc3


<h3 style='color:gray;'>Imports Playlist JSON file, using extra method (2/2)</h3>

In [6]:
def transform_playlist(data):
    all_playlists = []
    for i in data['playlists']:
        i_playlist = pd.json_normalize(i['items'], sep='_')
        i_playlist['playlists_lastModifiedDate'] = i['lastModifiedDate']
        i_playlist['playlists_name'] = i['name']
        all_playlists.append(i_playlist)
    return pd.concat(all_playlists)

playlist = import_json(PATH_PLAYLIST, transform=transform_playlist)
playlist.head(1)

,episode,audiobook,local_track,added_date,track_track_name,track_artist_name,track_album_name,track_track_uri,track,episode_episode_name,episode_show_name,episode_episode_uri,playlists_last_modified_date,playlists_name
0,NaN,None,None,2024-10-22,Angry Too,Lola Blanc,Angry Too,spotify:track:51jK7uI2QR8JCP4G9DnbQS,NaN,NaN,NaN,NaN,2026-01-18,<3


In [7]:
# 3. Convert lookup series to Python sets, before creating boolean columns
playlist_uris = set(playlist['track_track_uri'].dropna())
library_uris = set(library['uri'].dropna())

final_df['in_playlist'] = final_df['spotify_track_uri'].isin(playlist_uris)
final_df['in_library'] = final_df['spotify_track_uri'].isin(library_uris)

final_df.head(1)

,Unnamed: 0,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,spotify_episode_uri,audiobook_title,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode,artist_name,segment,track_id,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence,popularity,duration_ms,lat_long,temperature,precipitation,in_playlist,in_library
0,0,2020-10-20 19:36:53+00:00,"Android OS 10 API 29 (samsung, SM-A307GT)",23600,BR,177.58.181.120,Pretty Savage,THE ALBUM,spotify:track:1XnpzbOGptRwfJhZgLbmSr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,clickrow,backbtn,False,False,False,NaN,False,BLACKPINK,Previously Active Listeners,1XnpzbOGptRwfJhZgLbmSr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3",23.0,0.8,False,False


# Creation of Calculated Columns

<h3 style='color:gray;'>Creating Playlist Percentage (% of Song in All Playlists) [1/2]</h3>

In [8]:
#get total amount of playlist
playlist_amount = playlist['playlists_name'].nunique()

# 1. Get the counts of each value in df_2
counts = playlist['track_track_uri'].value_counts()

# 2. Map those counts to df_1
final_df['playlist_count'] = final_df['spotify_track_uri'].map(counts)

# 3. Fill NaNs with 0 (for values that didn't appear in df_2) and convert to integer
final_df['playlist_count'] = final_df['playlist_count'].fillna(0).astype(int)

final_df['percentage_playlists'] = final_df['playlist_count'] / playlist_amount

final_df.head(1)

,Unnamed: 0,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,spotify_episode_uri,audiobook_title,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode,artist_name,segment,track_id,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence,popularity,duration_ms,lat_long,temperature,precipitation,in_playlist,in_library,playlist_count,percentage_playlists
0,0,2020-10-20 19:36:53+00:00,"Android OS 10 API 29 (samsung, SM-A307GT)",23600,BR,177.58.181.120,Pretty Savage,THE ALBUM,spotify:track:1XnpzbOGptRwfJhZgLbmSr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,clickrow,backbtn,False,False,False,NaN,False,BLACKPINK,Previously Active Listeners,1XnpzbOGptRwfJhZgLbmSr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"-23.5,-46.3",23.0,0.8,False,False,0,0.0


<h3 style='color:gray;'>Adding Percentage Listening [3/3]</h3>

In [9]:
final_df['percentage_played'] = (final_df['ms_played'] / final_df['duration_ms']).round(4)

In [10]:
final_df.dropna(subset=['duration_ms']).head(1)

,Unnamed: 0,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,spotify_episode_uri,audiobook_title,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode,artist_name,segment,track_id,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence,popularity,duration_ms,lat_long,temperature,precipitation,in_playlist,in_library,playlist_count,percentage_playlists,percentage_played
9,9,2020-10-23 19:52:08+00:00,"Android OS 10 API 29 (samsung, SM-A307GT)",1064,BR,187.26.175.43,I'm Gonna Show You Crazy,I'm Gonna Show You Crazy,spotify:track:5LtNBCM2ve0SxP0dlRVvMu,NaN,NaN,NaN,NaN,NaN,NaN,NaN,clickrow,endplay,False,False,False,NaN,False,Bebe Rexha,Previously Active Listeners,5LtNBCM2ve0SxP0dlRVvMu,0.0758,0.474,0.537,0.0,2.0,0.316,-6.77,1.0,0.0649,90.103,0.335,56.0,207775.0,"-23.5,-46.6",22.4,0.1,True,False,1,1.0,0.0051


In [11]:
consolidated_selection = final_df.copy()

columns_to_keep_pre_transformation = [ 
    'ts',
    # 'ms_played',
    'acousticness',
    'danceability',
    'energy',
    'instrumentalness',
    'key',
    'liveness',
    'loudness',
    'mode',
    'speechiness',
    'tempo',
    'valence',
    'popularity',
    # 'duration_ms',
    'temperature',
    'precipitation',
    'in_playlist',
    'in_library',
    'playlist_count',
    'percentage_playlists',
    'percentage_played',
    'master_metadata_album_album_name',
    'reason_start',
    'reason_end',
    'shuffle',
    'skipped',
    'artist_name',
    'track_id',
    # 'lat_long',
]

# Keeping only the columns that will be used
consolidated_selection = consolidated_selection[columns_to_keep_pre_transformation]

# Dropping rows with any null cell
consolidated_selection = consolidated_selection.dropna()

consolidated_selection.head(3)

,ts,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence,popularity,temperature,precipitation,in_playlist,in_library,playlist_count,percentage_playlists,percentage_played,master_metadata_album_album_name,reason_start,reason_end,shuffle,skipped,artist_name,track_id
9,2020-10-23 19:52:08+00:00,0.0758,0.474,0.537,0.0,2.0,0.316,-6.77,1.0,0.0649,90.103,0.335,56.0,22.4,0.1,True,False,1,1.0,0.0051,I'm Gonna Show You Crazy,clickrow,endplay,False,False,Bebe Rexha,5LtNBCM2ve0SxP0dlRVvMu
10,2020-10-23 19:52:08+00:00,0.0758,0.474,0.537,0.0,2.0,0.316,-6.77,1.0,0.0649,90.103,0.335,56.0,22.4,0.1,True,False,1,1.0,0.0051,I'm Gonna Show You Crazy,clickrow,endplay,False,False,Bebe Rexha,5LtNBCM2ve0SxP0dlRVvMu
11,2020-10-23 19:55:36+00:00,0.0758,0.474,0.537,0.0,2.0,0.316,-6.77,1.0,0.0649,90.103,0.335,56.0,22.4,0.1,True,False,1,1.0,1.0000,I'm Gonna Show You Crazy,clickrow,trackdone,False,False,Bebe Rexha,5LtNBCM2ve0SxP0dlRVvMu


In [12]:
consolidated_selection['ts'] = pd.to_datetime(consolidated_selection['ts'])
hours = consolidated_selection['ts'].dt.hour

# Create columns as integers (1 or 0) from the jump
consolidated_selection['is_morning']   = hours.between(5, 11).astype(int)
consolidated_selection['is_afternoon'] = hours.between(12, 16).astype(int)
consolidated_selection['is_evening']   = hours.between(17, 20).astype(int)
consolidated_selection['is_night']     = ((hours >= 21) | (hours <= 4)).astype(int)
consolidated_selection = consolidated_selection.drop(columns=['ts'])
consolidated_selection.head(1)

,acousticness,danceability,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,valence,popularity,temperature,precipitation,in_playlist,in_library,playlist_count,percentage_playlists,percentage_played,master_metadata_album_album_name,reason_start,reason_end,shuffle,skipped,artist_name,track_id,is_morning,is_afternoon,is_evening,is_night
9,0.0758,0.474,0.537,0.0,2.0,0.316,-6.77,1.0,0.0649,90.103,0.335,56.0,22.4,0.1,True,False,1,1.0,0.0051,I'm Gonna Show You Crazy,clickrow,endplay,False,False,Bebe Rexha,5LtNBCM2ve0SxP0dlRVvMu,0,0,1,0


## Transformind & Cleaning Data

In [13]:
# Applies standard scaler
standard_cols = [
    'acousticness', 'danceability', 'energy', 'instrumentalness',
    'liveness', 'loudness', 'speechiness', 'tempo', 'valence',
    'popularity', 'playlist_count',
]

standard_scaler = StandardScaler()
consolidated_selection[standard_cols] = standard_scaler.fit_transform(consolidated_selection[standard_cols].astype(float))

In [14]:
# Applies min max
minmax_cols = ['temperature', 'precipitation']

minmax_scaler = MinMaxScaler()
consolidated_selection[minmax_cols] = minmax_scaler.fit_transform(consolidated_selection[minmax_cols].astype(float))

In [15]:
# Applies one hot encode
consolidated_selection['key'] = consolidated_selection['key'].astype(int)
key_dummies = pd.get_dummies(consolidated_selection['key'], prefix='key')

# Ensure all 12 columns exist even if some keys are absent in the data
for k in range(12):
    col = f'key_{k}'
    if col not in key_dummies.columns:
        key_dummies[col] = 0

key_dummies = key_dummies[[f'key_{k}' for k in range(12)]].astype(int)
consolidated_selection = pd.concat([consolidated_selection.drop(columns=['key']), key_dummies], axis=1)

In [16]:
# Applies binary transformation to boolean fields
binary_cols = [
    'mode', 'shuffle', 'skipped', 'in_playlist', 'in_library',
    'is_morning', 'is_afternoon', 'is_evening', 'is_night',
]

for col in binary_cols:
    consolidated_selection[col] = consolidated_selection[col].astype(bool).astype(int)

In [17]:
# Applies label encoding
label_encode_map = {
    'master_metadata_album_album_name': 'album_id',
    'artist_name':                      'artist_id',
    # 'track_id':                         'track_id_enc',
    'reason_start':                     'reason_start_enc',
    'reason_end':                       'reason_end_enc',
}

label_encoders = {}
for raw_col, enc_col in label_encode_map.items():
    le = LabelEncoder()
    consolidated_selection[enc_col] = le.fit_transform(consolidated_selection[raw_col].astype(str).fillna('unknown'))
    label_encoders[raw_col] = le


consolidated_selection.drop(columns=list(label_encode_map.keys()), inplace=True)

In [18]:
# Views transformed data
print(consolidated_selection.shape)
print(consolidated_selection.dtypes)

(8038, 41)
acousticness            float64
danceability            float64
energy                  float64
instrumentalness        float64
liveness                float64
loudness                float64
mode                      int64
speechiness             float64
tempo                   float64
valence                 float64
popularity              float64
temperature             float64
precipitation           float64
in_playlist               int64
in_library                int64
playlist_count          float64
percentage_playlists    float64
percentage_played       float64
shuffle                   int64
skipped                   int64
track_id                 object
is_morning                int64
is_afternoon              int64
is_evening                int64
is_night                  int64
key_0                     int64
key_1                     int64
key_2                     int64
key_3                     int64
key_4                     int64
key_5                     int

## Verifying & Saving Dataset

In [19]:
# Displays the data
consolidated_selection

,acousticness,danceability,energy,instrumentalness,liveness,loudness,mode,speechiness,tempo,valence,popularity,temperature,precipitation,in_playlist,in_library,playlist_count,percentage_playlists,percentage_played,shuffle,skipped,track_id,is_morning,is_afternoon,is_evening,is_night,key_0,key_1,key_2,key_3,key_4,key_5,key_6,key_7,key_8,key_9,key_10,key_11,album_id,artist_id,reason_start_enc,reason_end_enc
9,-0.713606,-0.814329,-0.674166,-0.123869,0.904744,-0.337662,1,-0.369621,-1.110379,-0.684905,-0.940824,0.458491,0.012346,1,0,1.049756,1.0,0.0051,0,0,5LtNBCM2ve0SxP0dlRVvMu,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,268,40,2,1
10,-0.713606,-0.814329,-0.674166,-0.123869,0.904744,-0.337662,1,-0.369621,-1.110379,-0.684905,-0.940824,0.458491,0.012346,1,0,1.049756,1.0,0.0051,0,0,5LtNBCM2ve0SxP0dlRVvMu,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,268,40,2,1
11,-0.713606,-0.814329,-0.674166,-0.123869,0.904744,-0.337662,1,-0.369621,-1.110379,-0.684905,-0.940824,0.458491,0.012346,1,0,1.049756,1.0,1.0000,0,0,5LtNBCM2ve0SxP0dlRVvMu,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,268,40,2,5
12,-0.713606,-0.814329,-0.674166,-0.123869,0.904744,-0.337662,1,-0.369621,-1.110379,-0.684905,-0.940824,0.458491,0.012346,1,0,1.049756,1.0,1.0000,0,0,5LtNBCM2ve0SxP0dlRVvMu,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,268,40,2,5
13,-1.007087,0.210728,0.267848,-0.123869,-0.821303,-0.117229,0,-0.736881,-0.878520,-1.261676,0.107015,0.458491,0.012346,1,0,1.049756,1.0,0.0801,0,0,04ZTP5KsCypmtCmQg5tH9R,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,188,40,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10927,-0.941186,0.782394,-0.374162,-0.123869,-0.959896,-0.563681,0,-0.657122,0.228331,0.328078,0.194335,0.326415,0.728395,1,0,1.049756,1.0,1.0012,1,0,2hdy9Wt9qp7M7d0U3ossu2,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,110,52,6,5
10928,-0.941186,0.782394,-0.374162,-0.123869,-0.959896,-0.563681,0,-0.657122,0.228331,0.328078,0.194335,0.326415,0.728395,1,0,1.049756,1.0,1.0012,1,0,2hdy9Wt9qp7M7d0U3ossu2,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,110,52,6,5
10929,-0.951640,-0.130958,0.861858,-0.123869,0.819891,0.422036,0,-0.551396,0.495764,0.783678,-0.504224,0.337736,0.000000,1,0,1.049756,1.0,0.0164,1,0,1R1ihosU2CSEaSw8e5ZF6E,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,37,170,6,3
10930,-0.951640,-0.130958,0.861858,-0.123869,0.819891,0.422036,0,-0.551396,0.495764,0.783678,-0.504224,0.337736,0.000000,1,0,1.049756,1.0,0.0164,1,0,1R1ihosU2CSEaSw8e5ZF6E,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,37,170,6,3


In [20]:
# Displays columns
consolidated_selection.columns

Index(['acousticness', 'danceability', 'energy', 'instrumentalness',
       'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence',
       'popularity', 'temperature', 'precipitation', 'in_playlist',
       'in_library', 'playlist_count', 'percentage_playlists',
       'percentage_played', 'shuffle', 'skipped', 'track_id', 'is_morning',
       'is_afternoon', 'is_evening', 'is_night', 'key_0', 'key_1', 'key_2',
       'key_3', 'key_4', 'key_5', 'key_6', 'key_7', 'key_8', 'key_9', 'key_10',
       'key_11', 'album_id', 'artist_id', 'reason_start_enc',
       'reason_end_enc'],
      dtype='object')

In [21]:
# Saves the dataset
consolidated_selection.to_csv(PATH_PROCESSED, index=False)